In [7]:
#%pip install llama-index --quiet
#%pip install gradientai --quiet
#%pip install llama-index-llms-gradient
%mkdir -p dataecho 

In [8]:
# Create a document containing information about 奥利奥, the channel's mascotte.s
with open('data/aoliao.txt', 'w') as f:
    f.write("奥利奥 is a 3 years old cat that teaches deep learning models on its YouTube channel along with his friend Umar Jamil.\n" +
            "They're both passionate about machine learning and deep learning, and 奥利奥 is very fast in learning new concepts.\n" +
            "So far, the duo has made videos on Large Language Models, Stable Diffusion and Transformer models, including the popular model LLaMA 2.\n" +
            "Apart from machine learning, 奥利奥 likes to play with his friend Umar, especially when he is recording videos for their YouTube channel.\n")

In [9]:
from llama_index.llms.gradient import GradientBaseModelLLM
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.gradient import GradientEmbedding

import os
# Paste values from Gradient's websites
os.environ["GRADIENT_ACCESS_TOKEN"] = "PASTE YOUR ACCESS TOKEN"
os.environ["GRADIENT_WORKSPACE_ID"] = "PASTE YOUR WORKSPACE ID"

question = "Do you know anyone named 奥利奥?"

# You can also use a model adapter you've trained with GradientModelAdapterLLM
llm = GradientBaseModelLLM(base_model_slug="llama2-7b-chat",max_tokens=100)

print(f'Without RAG: {llm.complete(question)}')
print(f'')

documents = SimpleDirectoryReader("./data").load_data() # Documents to index 
embed_model = GradientEmbedding(gradient_model_slug="bge-large") # The model used to generate embeddings
service_context = Settings.from_defaults(chunk_size=1024, llm=llm, embed_model=embed_model) # The service context defines the LLM and the embedding model to be used by the query engine

index = VectorStoreIndex.from_documents(documents, service_context=service_context)
query_engine = index.as_query_engine()

response = query_engine.query(question)
print(f'With RAG: {response}')

MaxRetryError: HTTPSConnectionPool(host='api.gradient.ai', port=443): Max retries exceeded with url: /api/models?capability=any&onlyBase=True (Caused by NameResolutionError("HTTPSConnection(host='api.gradient.ai', port=443): Failed to resolve 'api.gradient.ai' ([Errno 8] nodename nor servname provided, or not known)"))